# Range evaluation

We evaluate the per-hand **posterior over the target's latent hand class**, $q(h)$, that the EM E-step produces — that is, the *range* the model assigns to the target after seeing the action sequence in that hand.

Three ranges per held-out hand:

1. **Prior $\pi_0$** — observer-conditioned uniform over the 169 hand classes (`initial_class_prior(dead_cards)`). No actions used; this is the *floor*.
2. **Population posterior** — E-step using $\theta = (0,0,0)$, i.e. the global prior only.
3. **Player posterior** — E-step using the player's trained $\hat\theta$ from `player_thetas.json`.

Metrics:

* **True-hand NLL** $=-\log q(h^\star)$, where $h^\star$ is the target's actual hand class (from the `.phh` hole cards). Lower is better.
* **True-hand rank** — position of $h^\star$ in the 169-class list sorted by $q(h)$ descending (1 = best).
* **Top-$k$ accuracy** for $k \in \{1, 5, 10, 20, 50\}$ — fraction of hands where $h^\star$ is in the top-$k$ of $q$.

Scope: **preflop ranges only**. Postflop ranges live over 1326 specific combos and we only have ~16 postflop bundles per player — too thin for meaningful aggregate metrics. The preflop bundle set (78 per player) is the right size for this.

Test split: by default we evaluate on the **online split** (sessions never seen by the global-prior fit *and* never seen by the EM that fit $\theta$). We also report on the EM split for comparison; the EM split is the data $\theta$ was fit on, so it serves as the in-sample reference.


## Setup

In [2]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt


def _find_repo_root(start: Path) -> Path:
    for p in (start, *start.parents):
        if (p / "pipeline_common.py").is_file():
            return p
    raise FileNotFoundError(
        "Could not find pipeline_common.py; open this notebook from inside the repo."
    )


REPO_ROOT = _find_repo_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from pipeline_common import (
    flatten_hands,
    read_session_names_file,
    split_session_names,
    hole_cards_to_hand_class,
    preflop_decisions_for_hand,
)
from utils.prior.preflop import (
    PreflopPrior,
)
from utils.em.preflop import (
    PreflopEMDecision,
    PreflopEMHandBundle,
    e_step_hand_class_posterior,
)
from utils.filter.helpers import initial_class_prior, normalize
from utils.strength.preflop import all_169_classes

### Load $\beta$, $\theta$, and reconstruct splits

In [ ]:
priors = json.loads((REPO_ROOT / "artifacts" / "global_priors.json").read_text())
BETA_PRE = np.asarray(priors["preflop"]["beta_preflop"], dtype=float)

thetas_payload = json.loads((REPO_ROOT / "artifacts" / "player_thetas.json").read_text())
THETA_PRE = {p: tuple(thetas_payload["players"][p]["theta_pre"]) for p in thetas_payload["players"]}
PLAYERS = list(THETA_PRE.keys())
print("players:", PLAYERS)
for p in PLAYERS:
    print(f"  {p}: theta_pre = {tuple(round(x, 4) for x in THETA_PRE[p])}")

train_s = read_session_names_file(REPO_ROOT / "sessions_train.txt")
em_s = read_session_names_file(REPO_ROOT / "sessions_theta.txt")
online_s = read_session_names_file(REPO_ROOT / "sessions_filter.txt")


em_refs     = flatten_hands([REPO_ROOT / "pluribus" / s for s in em_s])
online_refs = flatten_hands([REPO_ROOT / "pluribus" / s for s in online_s])
print(f"em hands: {len(em_refs)}   online hands: {len(online_refs)}")

players: ['MrBlue', 'Bill', 'Pluribus']
  MrBlue: theta_pre = (-0.0421, 0.2102, -0.1681)
  Bill: theta_pre = (0.0722, -0.1918, 0.1196)
  Pluribus: theta_pre = (0.1714, -0.1727, 0.0013)


FileNotFoundError: [Errno 2] No such file or directory: '/Users/kevinliu/Desktop/bayesian poker/sessions.txt'

## Building $(bundle, h^\star)$ pairs

We re-create one preflop bundle per `(hand, observer, target)` triple, just like `gather_preflop_bundles_for_target_player`, but also capture the **true hand class** $h^\star$ from the target's hole cards so we can score the resulting posterior.

In [ ]:
def build_bundles_with_truth(refs, target: str, observer: str):
    """Returns list of (PreflopEMHandBundle, true_class, meta)."""
    rows = []
    for ref in refs:
        names = ref.hand.player_names
        if target not in names or observer not in names:
            continue
        true_class = hole_cards_to_hand_class(ref.hand.hole_cards.get(target, "") or "")
        if true_class is None:
            continue
        decisions = preflop_decisions_for_hand(ref.hand, target, ref.global_index)
        if not decisions:
            continue
        dead = ref.hand.hole_cards.get(observer, "") or ""
        initial_range = normalize(initial_class_prior(dead_cards=dead))
        bundle = PreflopEMHandBundle(
            tuple(PreflopEMDecision(d.state_key, d.action_bucket) for d in decisions),
            initial_range,
        )
        rows.append((bundle, true_class, {
            "global_index": ref.global_index,
            "n_decisions": len(decisions),
        }))
    return rows

# Pluribus is the target; Gogo is the observer  (and vice versa)
OBSERVER_OF = {
    "Gogo": "Pluribus",
    "Pluribus": "Gogo",
}

DATA = {}  # DATA[(player, split)] = list of (bundle, true_class, meta)
for player in PLAYERS:
    observer = OBSERVER_OF[player]
    DATA[(player, "em")]     = build_bundles_with_truth(em_refs,     player, observer)
    DATA[(player, "online")] = build_bundles_with_truth(online_refs, player, observer)

for (p, sp), rows in DATA.items():
    print(f"{p:<10} {sp:<6} bundles: {len(rows)}")

## Running the E-step under three priors

For each bundle we compute the posterior $q(h)$ under three settings of the prior:

* **prior_only** — skip the E-step entirely; return `bundle.initial_range`. This is $\pi_0$, the *observer-conditioned* uniform.
* **population** — `PreflopPrior(theta_pre=(0,0,0), beta_preflop=BETA_PRE)`. Uses actions but only with the population baseline.
* **player** — `PreflopPrior(theta_pre=THETA_PRE[player], beta_preflop=BETA_PRE)`. Uses actions and the player's fitted tilt.


In [ ]:
def run_estep(rows, prior):
    return [(true_class, e_step_hand_class_posterior(bundle, prior))
            for bundle, true_class, _meta in rows]

def prior_only_results(rows):
    return [(true_class, dict(bundle.initial_range)) for bundle, true_class, _meta in rows]

RESULTS = {}  # (player, split, predictor) -> list[(true_class, q_dict)]
for player in PLAYERS:
    prior_pop = PreflopPrior(theta_pre=(0.0, 0.0, 0.0), beta_preflop=BETA_PRE)
    prior_pl  = PreflopPrior(theta_pre=THETA_PRE[player], beta_preflop=BETA_PRE)
    for split in ("em", "online"):
        rows = DATA[(player, split)]
        RESULTS[(player, split, "prior_only")] = prior_only_results(rows)
        RESULTS[(player, split, "population")] = run_estep(rows, prior_pop)
        RESULTS[(player, split, "player")]     = run_estep(rows, prior_pl)

print("finished computing posteriors for", len(RESULTS), "settings")

## Metric 1 — True-hand NLL

$$\mathrm{NLL} = -\frac{1}{N}\sum_{i=1}^{N}\log q_i(h^\star_i),$$

averaged over bundles. Lower is better. Floored at $10^{-12}$ so a hand class with zero posterior mass doesn't break the metric.

In [ ]:
EPS = 1e-12

def true_hand_nll(results):
    if not results:
        return float("nan")
    nlls = []
    for true_class, q in results:
        p = float(q.get(true_class, 0.0))
        nlls.append(-np.log(max(p, EPS)))
    return float(np.mean(nlls))

print(f"{'player':<10} {'split':<7} {'predictor':<11} {'NLL':>8}")
for player in PLAYERS:
    for split in ("em", "online"):
        for predictor in ("prior_only", "population", "player"):
            v = true_hand_nll(RESULTS[(player, split, predictor)])
            print(f"{player:<10} {split:<7} {predictor:<11} {v:>8.4f}")

## Metric 2 — True-hand rank

Rank of $h^\star$ in $q$ sorted descending. Rank 1 = the true hand is the modal prediction; rank 169 = the model literally cannot identify the hand at all.

In [ ]:
ALL_169 = all_169_classes()
ALL_169_INDEX = {h: i for i, h in enumerate(ALL_169)}

def ranks(results):
    out = []
    for true_class, q in results:
        # full 169-class vector; missing classes -> mass 0
        p = np.array([q.get(h, 0.0) for h in ALL_169], dtype=float)
        order = np.argsort(-p, kind="stable")
        rank = int(np.where(order == ALL_169_INDEX[true_class])[0][0]) + 1
        out.append(rank)
    return np.asarray(out, dtype=int)

RANKS = {key: ranks(res) for key, res in RESULTS.items()}

print(f"{'player':<10} {'split':<7} {'predictor':<11} {'mean_rank':>10} {'median_rank':>12}")
for key, r in RANKS.items():
    if r.size == 0:
        continue
    p, sp, pred = key
    print(f"{p:<10} {sp:<7} {pred:<11} {r.mean():>10.2f} {np.median(r):>12.1f}")

## Metric 3 — Top-$k$ accuracy

Fraction of bundles whose true class is in the top-$k$ of $q$. Coarser than rank itself but easier to interpret as a hit-rate. Random baseline at $k$ would give roughly $k/169$ if $\pi_0$ were truly uniform; the observer's dead cards tilt it slightly.

In [ ]:
KS = (1, 5, 10, 20, 50)

def topk_table(rank_arrays_by_key):
    header = ["player", "split", "predictor"] + [f"top-{k}" for k in KS]
    print("  ".join(f"{h:<10}" for h in header))
    for (p, sp, pred), r in rank_arrays_by_key.items():
        if r.size == 0:
            continue
        cells = [p, sp, pred] + [f"{(r <= k).mean():.3f}" for k in KS]
        print("  ".join(f"{c:<10}" for c in cells))

topk_table(RANKS)

## Rank distribution plots

ECDFs of the true-hand rank under each predictor. A predictor that pushes more mass to the left has more probability on the true hand. Compare prior-only (no actions used) to the two action-informed predictors.

In [ ]:
def ecdf(arr):
    x = np.sort(arr)
    y = np.arange(1, x.size + 1) / x.size
    return x, y

fig, axes = plt.subplots(len(PLAYERS), 2, figsize=(11, 4 * len(PLAYERS)), sharex=True, sharey=True)
if len(PLAYERS) == 1:
    axes = np.array([axes])
for i, player in enumerate(PLAYERS):
    for j, split in enumerate(("em", "online")):
        ax = axes[i, j]
        for predictor, ls in (("prior_only", ":"), ("population", "--"), ("player", "-")):
            r = RANKS[(player, split, predictor)]
            if r.size == 0:
                continue
            x, y = ecdf(r)
            ax.step(x, y, where="post", label=predictor, linestyle=ls)
        ax.set_title(f"{player}  ({split} split, n={RANKS[(player, split, 'player')].size})")
        ax.set_xlabel("true-hand rank in q")
        ax.set_ylabel("ECDF")
        ax.set_xlim(1, 169)
        ax.legend(loc="lower right")
        ax.grid(alpha=0.3)
fig.suptitle("ECDF of the true hand's rank in the predicted range")
fig.tight_layout()
plt.show()

## Reading the results

* **prior_only** is the floor: it uses no action information. If `population` or `player` does not beat it on NLL and rank, the prior is doing all of the work and the action model is not contributing identifying signal.
* **population** vs **player**: this is the *value of the per-player $\theta$*. A gap (player beats population) means $\hat\theta$ has captured a real per-player deviation that translates into better hand inference. No gap means $\hat\theta$ is too small or too uncertain to move the range materially — usually a data-quantity problem at this scale.
* The **EM split** is where $\theta$ was fit; expect the player-tilted posterior to look best there. The **online split** is the honest generalisation check.
* Mean rank around the uniform expectation $\sim 85$ on a 169-class range means the model is providing no information beyond the prior. Mean rank around $\sim 40$ would mean the posterior is roughly 2× sharper than uniform on average — a meaningful win at this latent-space size.
* Top-1 accuracy is almost always disappointing here (the true hand has $\sim 1\%$ prior mass before actions). Top-20 / top-50 are the realistic operating points.